In [1]:
import os, sys, glob
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

WORK = "/kaggle/working"
DATA_DIR = f"{WORK}/data"
OUT_DIR = f"{WORK}/outputs"
RESULTS_DIR = f"{WORK}/results"
for d in (DATA_DIR, OUT_DIR, RESULTS_DIR):
    os.makedirs(d, exist_ok=True)

def find(name):
    hits = sorted(glob.glob(f"{WORK}/**/{name}", recursive=True)) + \
           sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"{name} not found: attach the dataset/notebook that makes it (Add Input)")
    if len(hits) > 1:
        print(f"note: {len(hits)} copies of {name}, using {hits[0]}")
    return hits[0]

RAW_CSV = find("afri_med_qa_15k_v2.4_phase_2_15275.csv")

HF_USER = "Tresor26"
def hub_id(run_name):
    return f"{HF_USER}/clinicmate-{run_name}"

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret("HF_TOKEN"))

REPO = "/kaggle/temp/ClinicMate"
if not os.path.exists(REPO):
    !git clone -q https://github.com/Tresorshingiro/ClinicMate.git {REPO}
else:
    !git -C {REPO} pull -q
sys.path.append(REPO)

import random, numpy as np
SEED = 42
random.seed(SEED); np.random.seed(SEED)

In [7]:
!pip install -q evaluate rouge_score bert_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.6 MB/s eta 0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.7 MB/s eta 0:00:00


In [5]:
import json, numpy as np, pandas as pd
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from src.evaluation import text_metrics, bootstrap_ci

In [8]:
train = pd.read_json(find("train.jsonl"), lines=True)
ev = pd.read_json(find("test_eval.jsonl"), lines=True)
results = {}
print(len(train), len(ev))

3663 982


In [10]:
mcq = ev[ev.qtype == "mcq"]
gold = mcq["answer_letters"].str[0].tolist()
rng = np.random.default_rng(42)
rand = ["ABCDE"[rng.integers(len(o))] for o in mcq["options"]]
maj_letter = Counter(train[train.qtype == "mcq"]["answer_letters"].explode()).most_common(1)[0][0]
for name, preds in [("B0_random", rand), ("B0_majority", [maj_letter] * len(gold))]:
    correct = [p == g for p, g in zip(preds, gold)]
    results[name] = {"mcq_acc": float(np.mean(correct)), "mcq_ci": bootstrap_ci(correct)}
print(results)

{'B0_random': {'mcq_acc': 0.2124492557510149, 'mcq_ci': (0.18267929634641408, 0.2422192151556157)}, 'B0_majority': {'mcq_acc': 0.22868741542625168, 'mcq_ci': (0.19888362652232747, 0.2598105548037889)}}


In [16]:
saq_train = train[train.qtype == "saq"].reset_index(drop=True)
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, stop_words="english", sublinear_tf=True)
X_train = vec.fit_transform(saq_train["question"])
sub = ev[ev.qtype == "saq"]
sims = vec.transform(sub["question"]) @ X_train.T
nn = np.asarray(sims.argmax(axis=1)).ravel()
preds = saq_train.loc[nn, "rationale"].tolist()
m = text_metrics(preds, sub["rationale"].tolist())
results["B1_tfidf_retrieval"] = {"saq": {k: m[k] for k in ["rougeL", "bertscore_f1"]}}

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [17]:
json.dump(results, open(f"{RESULTS_DIR}/baselines.json", "w"), indent=2)
print(json.dumps(results, indent=2))

{
  "B0_random": {
    "mcq_acc": 0.2124492557510149,
    "mcq_ci": [
      0.18267929634641408,
      0.2422192151556157
    ]
  },
  "B0_majority": {
    "mcq_acc": 0.22868741542625168,
    "mcq_ci": [
      0.19888362652232747,
      0.2598105548037889
    ]
  },
  "B1_tfidf_retrieval": {
    "saq": {
      "rougeL": 0.2579368997778502,
      "bertscore_f1": 0.2086481056003659
    }
  }
}
